# Experiments

- On the terminal: Go to project folder, activate VM

- Start `mlflow` on terminal

```bash
mlflow ui \
  --backend-store-uri sqlite:///../mlflow_db/mlflow.db \
  --default-artifact-root ../mlruns \
  --host 127.0.0.1 \
  --port 5001
```

In [4]:
# basic setup and import
import os
import json
import tempfile
import numpy as np
import pandas as pd

import mlflow
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

# ----------------------------
# MLflow experiment
# ----------------------------
mlflow.set_tracking_uri("sqlite:///../mlflow_db/mlflow.db")
print("Tracking URI:", mlflow.get_tracking_uri())


Tracking URI: sqlite:///../mlflow_db/mlflow.db


In [5]:
mlflow.set_experiment("heart_disease_kaggle")

<Experiment: artifact_location='/Users/gabi/codes/kaggle/02_predicting_heart_disease/mlruns/1', creation_time=1770636741299, experiment_id='1', last_update_time=1770636741299, lifecycle_stage='active', name='heart_disease_kaggle', tags={}>

In [8]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
print([e.name for e in client.search_experiments()])

['heart_disease_kaggle', 'Default']


In [43]:
# treat dataset
def treat_dataset(data):
    # rename column names for easier access
    data.columns = [col.lower().replace(' ', '_') for col in data.columns]

    # convert heart_disease to binary
    if 'heart_disease' in data.columns:
        data['heart_disease'] = data['heart_disease'].apply(lambda x: 0 if (x == "Absence" or x == 0) else 1)

    return data


In [7]:
# read data file and prepare dataset
target = "heart_disease"

data = pd.read_csv("data/train.csv")

train_data = treat_dataset(data)

## CatBoost Experimentation

In [ ]:
from catboost import CatBoostClassifier, Pool

# ----------------------------
# Data prep: drop id, split holdout once
# ----------------------------
df = train_data.drop(columns=["id"]).copy()
X = df[features].copy()
y = df[target].astype(int).copy()

cat_idx = [X.columns.get_loc(c) for c in categorical_features]

X_tr, X_ho, y_tr, y_ho = train_test_split(
    X, y,
    test_size=0.15,
    stratify=y,
    random_state=42
)


# ----------------------------
# CV evaluation function (returns fold metrics + best iters)
# ----------------------------
def cv_catboost_auc(params, X_train, y_train, cat_idx, n_splits=5, seed=42):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_aucs = []
    fold_best_iters = []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train, y_train), 1):
        X_trf, X_vaf = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_trf, y_vaf = y_train.iloc[tr_idx], y_train.iloc[va_idx]

        train_pool = Pool(X_trf, y_trf, cat_features=cat_idx)
        valid_pool = Pool(X_vaf, y_vaf, cat_features=cat_idx)

        model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            allow_writing_files=False,
            verbose=False,
            od_type="Iter",
            od_wait=int(params["od_wait"]),
            iterations=int(params["iterations"]),
            learning_rate=float(params["learning_rate"]),
            depth=int(params["depth"]),
            l2_leaf_reg=float(params["l2_leaf_reg"]),
            subsample=float(params["subsample"]),
            rsm=float(params["rsm"]),
            min_data_in_leaf=int(params["min_data_in_leaf"]), # Optional: if you want more regularization control
        )

        model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

        p = model.predict_proba(valid_pool)[:, 1]
        auc = roc_auc_score(y_vaf, p)

        fold_aucs.append(auc)
        fold_best_iters.append(model.get_best_iteration())

    return fold_aucs, fold_best_iters

# ----------------------------
# Hyperopt search space
# ----------------------------
space = {
    # big upper bound; early stopping selects best iteration per fold
    "iterations": hp.quniform("iterations", 1000, 6000, 250),
    "learning_rate": hp.loguniform("learning_rate", np.log(0.01), np.log(0.2)),
    "depth": hp.quniform("depth", 4, 10, 1),
    "l2_leaf_reg": hp.loguniform("l2_leaf_reg", np.log(1.0), np.log(20.0)),
    "subsample": hp.uniform("subsample", 0.6, 1.0),
    "rsm": hp.uniform("rsm", 0.6, 1.0),
    "od_wait": hp.quniform("od_wait", 100, 400, 50),
    "min_data_in_leaf": hp.quniform("min_data_in_leaf", 5, 80, 5),
}

# ----------------------------
# Objective function: logs each trial into MLflow
# ----------------------------
def objective(params):
    # Cast hyperopt outputs
    params = dict(params)
    params["iterations"] = int(params["iterations"])
    params["depth"] = int(params["depth"])
    params["od_wait"] = int(params["od_wait"])
    params["min_data_in_leaf"] = int(params["min_data_in_leaf"])

    with mlflow.start_run(nested=True):
        # Log params
        mlflow.log_params(params)

        # Log feature config as artifact (json)
        feat_payload = {
            "numeric_features": numeric_features,
            "categorical_features": categorical_features,
            "all_features": features
        }
        with tempfile.TemporaryDirectory() as tmpdir:
            feat_path = os.path.join(tmpdir, "features.json")
            with open(feat_path, "w") as f:
                json.dump(feat_payload, f, indent=2)
            mlflow.log_artifact(feat_path, artifact_path="config")

        # CV
        fold_aucs, fold_best_iters = cv_catboost_auc(params, X_tr, y_tr, cat_idx, n_splits=10, seed=42)
        mean_auc = float(np.mean(fold_aucs))
        std_auc = float(np.std(fold_aucs))
        med_best_iter = int(np.median(fold_best_iters))

        # Log CV metrics
        mlflow.log_metric("cv_auc_mean", mean_auc)
        mlflow.log_metric("cv_auc_std", std_auc)
        mlflow.log_metric("cv_best_iter_median", med_best_iter)

        for i, auc in enumerate(fold_aucs, 1):
            mlflow.log_metric(f"cv_auc_fold_{i}", float(auc))
        for i, bi in enumerate(fold_best_iters, 1):
            mlflow.log_metric(f"cv_best_iter_fold_{i}", int(bi))

        # Train final model on full TRAIN split using median best_iter
        train_pool_full = Pool(X_tr, y_tr, cat_features=cat_idx)
        holdout_pool = Pool(X_ho, y_ho, cat_features=cat_idx)

        final_model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=42,
            allow_writing_files=False,
            verbose=False,
            iterations=med_best_iter,
            learning_rate=float(params["learning_rate"]),
            depth=int(params["depth"]),
            l2_leaf_reg=float(params["l2_leaf_reg"]),
            subsample=float(params["subsample"]),
            rsm=float(params["rsm"]),
        )

        final_model.fit(train_pool_full)

        # Holdout AUC
        p_ho = final_model.predict_proba(holdout_pool)[:, 1]
        ho_auc = float(roc_auc_score(y_ho, p_ho))

        mlflow.log_metric("holdout_auc", ho_auc)

        # Save model artifact
        with tempfile.TemporaryDirectory() as tmpdir:
            model_path = os.path.join(tmpdir, "catboost_model.cbm")
            final_model.save_model(model_path)
            mlflow.log_artifact(model_path, artifact_path="model")

        # Hyperopt minimizes, so return negative AUC
        return {"loss": -mean_auc, "status": STATUS_OK, "cv_auc_mean": mean_auc, "holdout_auc": ho_auc}

# ----------------------------
# Run the search (top-level MLflow run)
# ----------------------------
max_evals = 25
trials = Trials()

with mlflow.start_run(run_name="catboost_hyperopt"):
    mlflow.log_param("max_evals", max_evals)
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=max_evals,
        trials=trials,
        rstate=np.random.default_rng(42),
    )

print("Best hyperopt space (raw):", best)

# Optional: extract the best trial details
best_trial = min(trials.results, key=lambda r: r["loss"])
print("Best CV AUC:", best_trial["cv_auc_mean"])
print("Best Holdout AUC:", best_trial["holdout_auc"])


### Fast version

In [10]:
# ============================
# CatBoost + Hyperopt + MLflow (FAST version)
# Logs: features, CV AUC per fold, CV mean/std, best_iteration per fold, holdout AUC, model artifact
# ============================

from catboost import CatBoostClassifier, Pool


# ----------------------------
# 1) Define features
# ----------------------------
target = "heart_disease"

numeric_features = ["age", "bp", "cholesterol", "max_hr", "st_depression", "number_of_vessels_fluro"]
categorical_features = ["sex", "fbs_over_120", "exercise_angina", "chest_pain_type", "ekg_results", "slope_of_st", "thallium"]
features = numeric_features + categorical_features


# ----------------------------
# 2) Data prep: drop id, split holdout once
# ----------------------------
df = train_data.drop(columns=["id"]).copy()
X = df[features].copy()
y = df[target].astype(int).copy()

cat_idx = [X.columns.get_loc(c) for c in categorical_features]

X_tr, X_ho, y_tr, y_ho = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)


# ----------------------------
# 3) Pre-create CV Pools once (big speed win)
# ----------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_pools = []
for tr_idx, va_idx in cv.split(X_tr, y_tr):
    train_pool = Pool(X_tr.iloc[tr_idx], y_tr.iloc[tr_idx], cat_features=cat_idx)
    valid_pool = Pool(X_tr.iloc[va_idx], y_tr.iloc[va_idx], cat_features=cat_idx)
    fold_pools.append((train_pool, valid_pool))


# ----------------------------
# 4) Fast CV eval using pre-built Pools
# ----------------------------
def cv_catboost_auc_fast(params, fold_pools, seed=42):
    fold_aucs = []
    fold_best_iters = []

    for train_pool, valid_pool in fold_pools:
        model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            allow_writing_files=False,
            verbose=False,

            od_type="Iter",
            od_wait=int(params["od_wait"]),

            iterations=int(params["iterations"]),
            learning_rate=float(params["learning_rate"]),
            depth=int(params["depth"]),
            l2_leaf_reg=float(params["l2_leaf_reg"]),
            subsample=float(params["subsample"]),
            rsm=float(params["rsm"]),
            min_data_in_leaf=int(params["min_data_in_leaf"]),

            thread_count=-1,
            bootstrap_type="Bernoulli",
        )

        model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

        p = model.predict_proba(valid_pool)[:, 1]
        auc = roc_auc_score(valid_pool.get_label(), p)

        fold_aucs.append(float(auc))
        fold_best_iters.append(int(model.get_best_iteration()))

    return fold_aucs, fold_best_iters


# ----------------------------
# 5) Hyperopt search space (reduced for speed; early stopping handles exact #trees)
# ----------------------------
space = {
    "iterations": hp.quniform("iterations", 800, 3000, 200),
    "learning_rate": hp.loguniform("learning_rate", np.log(0.02), np.log(0.15)),
    "depth": hp.quniform("depth", 4, 9, 1),
    "l2_leaf_reg": hp.loguniform("l2_leaf_reg", np.log(1.0), np.log(20.0)),
    "subsample": hp.uniform("subsample", 0.7, 1.0),
    "rsm": hp.uniform("rsm", 0.7, 1.0),
    "od_wait": hp.quniform("od_wait", 80, 250, 30),
    "min_data_in_leaf": hp.quniform("min_data_in_leaf", 20, 120, 10),
}


# ----------------------------
# 6) Objective: logs each trial into MLflow
# ----------------------------
def objective(params):
    params = dict(params)
    params["iterations"] = int(params["iterations"])
    params["depth"] = int(params["depth"])
    params["od_wait"] = int(params["od_wait"])
    params["min_data_in_leaf"] = int(params["min_data_in_leaf"])

    with mlflow.start_run(nested=True):
        mlflow.log_params(params)

        # Log feature config as artifact
        feat_payload = {
            "numeric_features": numeric_features,
            "categorical_features": categorical_features,
            "all_features": features
        }
        with tempfile.TemporaryDirectory() as tmpdir:
            feat_path = os.path.join(tmpdir, "features.json")
            with open(feat_path, "w") as f:
                json.dump(feat_payload, f, indent=2)
            mlflow.log_artifact(feat_path, artifact_path="config")

        # CV (5 folds)
        fold_aucs, fold_best_iters = cv_catboost_auc_fast(params, fold_pools, seed=42)
        mean_auc = float(np.mean(fold_aucs))
        std_auc = float(np.std(fold_aucs))
        med_best_iter = int(np.median(fold_best_iters))

        # Log CV metrics
        mlflow.log_metric("cv_auc_mean", mean_auc)
        mlflow.log_metric("cv_auc_std", std_auc)
        mlflow.log_metric("cv_best_iter_median", med_best_iter)

        for i, auc in enumerate(fold_aucs, 1):
            mlflow.log_metric(f"cv_auc_fold_{i}", float(auc))
        for i, bi in enumerate(fold_best_iters, 1):
            mlflow.log_metric(f"cv_best_iter_fold_{i}", int(bi))

        # Holdout model (train on ALL train split using median best_iter)
        train_pool_full = Pool(X_tr, y_tr, cat_features=cat_idx)
        holdout_pool = Pool(X_ho, y_ho, cat_features=cat_idx)

        final_model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=42,
            allow_writing_files=False,
            verbose=False,

            iterations=med_best_iter,
            learning_rate=float(params["learning_rate"]),
            depth=int(params["depth"]),
            l2_leaf_reg=float(params["l2_leaf_reg"]),
            subsample=float(params["subsample"]),
            rsm=float(params["rsm"]),
            min_data_in_leaf=int(params["min_data_in_leaf"]),

            thread_count=-1,
            bootstrap_type="Bernoulli",
        )

        final_model.fit(train_pool_full)

        p_ho = final_model.predict_proba(holdout_pool)[:, 1]
        ho_auc = float(roc_auc_score(y_ho, p_ho))
        mlflow.log_metric("holdout_auc", ho_auc)

        # Save model artifact
        with tempfile.TemporaryDirectory() as tmpdir:
            model_path = os.path.join(tmpdir, "catboost_model.cbm")
            final_model.save_model(model_path)
            mlflow.log_artifact(model_path, artifact_path="model")

        return {"loss": -mean_auc, "status": STATUS_OK, "cv_auc_mean": mean_auc, "holdout_auc": ho_auc}


# ----------------------------
# 7) Run search
# ----------------------------
max_evals = 5  # start with 3 to validate logging/speed, then increase
trials = Trials()

with mlflow.start_run(run_name="catboost_hyperopt_fast_v2"):
    mlflow.log_param("max_evals", max_evals)

    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=max_evals,
        trials=trials,
        rstate=np.random.default_rng(42),
    )

print("Best hyperopt space (raw):", best)

best_trial = min(trials.results, key=lambda r: r["loss"])
print("Best CV AUC:", best_trial["cv_auc_mean"])
print("Best Holdout AUC:", best_trial["holdout_auc"])


100%|██████████| 5/5 [2:18:10<00:00, 1658.14s/trial, best loss: -0.9553155397987029]  
Best hyperopt space (raw): {'depth': np.float64(4.0), 'iterations': np.float64(2800.0), 'l2_leaf_reg': np.float64(1.2669504094959398), 'learning_rate': np.float64(0.06232209125475877), 'min_data_in_leaf': np.float64(30.0), 'od_wait': np.float64(240.0), 'rsm': np.float64(0.8824747486513409), 'subsample': np.float64(0.7901264064371525)}
Best CV AUC: 0.9553155397987029
Best Holdout AUC: 0.956499342407204


In [11]:
# load catboost model artifact from best trial
best_run = mlflow.search_runs(filter_string='tags.mlflow.runName = "catboost_hyperopt_fast_v2"', order_by=["metrics.cv_auc_mean DESC"], max_results=1)
best_run_id = best_run.iloc[0].run_id
print("Best run ID:", best_run_id)

Best run ID: 505e71d3a40a4332acc387bad69ac39a


In [34]:
# load cbm file
cbm_path = "/Users/gabi/codes/kaggle/02_predicting_heart_disease/mlruns/1/04ad3a355f634f6790fea58106d39d42/artifacts/model/catboost_model.cbm"
model_bestCatBoostSearch = CatBoostClassifier()
model_bestCatBoostSearch.load_model(cbm_path)

print("Loaded model from:", cbm_path)

Loaded model from: /Users/gabi/codes/kaggle/02_predicting_heart_disease/mlruns/1/04ad3a355f634f6790fea58106d39d42/artifacts/model/catboost_model.cbm


In [35]:
# load holdout data and evaluate loaded model
holdout_data = pd.read_csv("data/holdout_split.csv")
holdout_data = treat_dataset(holdout_data)


In [36]:
df = holdout_data.drop(columns=["id"]).copy()
X = df[features].copy()
y = df[target].astype(int).copy()

cat_idx = [X.columns.get_loc(c) for c in categorical_features]

In [37]:
holdout_pool_final = Pool(X, y, cat_features=cat_idx)

p_holdout_final = model_bestCatBoostSearch.predict_proba(holdout_pool_final)[:, 1]

final_holdout_auc = roc_auc_score(y, p_holdout_final)

print("Final Holdout AUC from loaded model:", final_holdout_auc)

Final Holdout AUC from loaded model: 0.9565181249145097


In [38]:
# load first catboost model from file

path = "/Users/gabi/codes/kaggle/02_predicting_heart_disease/models/catboost_heart_disease.cbm"
model_CatBoostManual = CatBoostClassifier()
model_CatBoostManual.load_model(path)

print("Loaded model from:", path)

Loaded model from: /Users/gabi/codes/kaggle/02_predicting_heart_disease/models/catboost_heart_disease.cbm


In [39]:
holdout_data = pd.read_csv("data/holdout_split.csv")
holdout_data = treat_dataset(holdout_data)

df = holdout_data.drop(columns=["id"]).copy()
X = df[features].copy()
y = df[target].astype(int).copy()

cat_idx = [X.columns.get_loc(c) for c in categorical_features]


holdout_pool_final = Pool(X, y, cat_features=cat_idx)

p_holdout_final = model_CatBoostManual.predict_proba(holdout_pool_final)[:, 1]

final_holdout_auc = roc_auc_score(y, p_holdout_final)

print("Final Holdout AUC from first Catboost model:", final_holdout_auc)

Final Holdout AUC from first Catboost model: 0.9567676519950403


In [40]:
# check parameters
model_CatBoostManual.get_params()

{'allow_writing_files': False,
 'eval_metric': 'AUC',
 'verbose': 200,
 'iterations': 1993,
 'loss_function': 'Logloss',
 'l2_leaf_reg': 3,
 'depth': 6,
 'random_seed': 42,
 'learning_rate': 0.03}

In [41]:
# check parameters from grid search catboost model
model_bestCatBoostSearch.get_params()

{'rsm': 0.8824747487,
 'allow_writing_files': False,
 'bootstrap_type': 'Bernoulli',
 'eval_metric': 'AUC',
 'verbose': 0,
 'iterations': 1836,
 'loss_function': 'Logloss',
 'l2_leaf_reg': 1.266950409,
 'subsample': 0.7901264064,
 'depth': 4,
 'min_data_in_leaf': 30,
 'learning_rate': 0.06232209125,
 'random_seed': 42}

In [44]:
# run CatBoostSearch to generate predictions for Kaggle submission
kaggle = pd.read_csv("data/test.csv")
kaggle = treat_dataset(kaggle)

df = kaggle.drop(columns=["id"]).copy()
X = df[features].copy()

cat_idx = [X.columns.get_loc(c) for c in categorical_features]
kaggle_pool = Pool(X, cat_features=cat_idx)
p_kaggle = model_bestCatBoostSearch.predict_proba(kaggle_pool)[:, 1]

submission = pd.DataFrame({
    "id": kaggle["id"],
    "heart_disease": p_kaggle
})

submission.to_csv("catboost_hyperopt_fast_v2_submission.csv", index=False)
print("Saved submission file: catboost_hyperopt_fast_v2_submission.csv")


Saved submission file: catboost_hyperopt_fast_v2_submission.csv


## Logistic Regression Experimentation